In [ ]:
# 1) Imports and utilities
import copy
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.distributions import Normal, Categorical
from typing import Tuple

# reproducibility
def set_seed(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 2) Offline dataset loader
A simple Dataset wrapper for offline RL (states, actions, rewards, next_states, dones). Actions can be continuous or discrete.

## Lecture 24 — Detailed Notes (expanded)

**1 The Generalisation Gap**

Agents trained on simulators can perform spectacularly in-domain yet fail when deployed offline in new situations. Offline RL asks: given a fixed logged dataset D drawn from some behavior policy β, can we learn a near-optimal policy without further interaction? The key practical motivation includes robotics, surgical settings, and large-scale autonomous fleets where further data collection is expensive or unsafe.

**2 Differences from Supervised Learning**

- Data is not i.i.d.; trajectories are temporally correlated and reflect a behaviour policy distribution d_β.
- Evaluation is expensive because environment interaction is required for validation.
- The learner must reason about counterfactuals: Q(s,a) for (s,a) not present in D.

**3 Offline RL Setting (Formal)**

Given D = {(s_i, a_i, r_i, s'_i)} sampled from discounted occupancy of β, the goal is to produce π^⋄ ≈ argmax_π J(π) without further environment interaction.

**4 Distribution Shift**

Define supp(D) = {(s,a) : (s,a) ∈ D}. An (s,a) pair is OOD if it ∉ supp(D). Offline methods aim either to keep π inside supp(D) (policy constraints) or to remove OOD actions from bootstrapping (algorithmic changes).

**5 A Taxonomy of Data Regimes**

- Imitation-like: near-expert trajectories — learning is mainly aggregation and denoising.
- Mixed-quality / heterogeneous-skill: both good and suboptimal trajectories — the agent must identify and prefer good behaviors (stitching).
- Fragmented data: many short fragments, heavy stitching required.

**6 Why Offline RL is Hard**

- Counterfactual evaluation: Q(s,a) for unseen actions is unobserved and may be arbitrarily inaccurate.
- Bootstrapping amplification: small OOD errors can amplify geometrically under repeated backups, causing massive overestimation.

**7 Key Algorithmic Principles**

All modern ORL methods follow two principles:
1. Keep π close to β (explicit or implicit constraints).
2. Modify learning to avoid bootstrapping on OOD actions (IQL-style expectiles, conservative Q estimates).

**8 Overview of Methods Implemented in this Notebook**

- Explicit policy constraint: KL penalties and reward-shaping (r - λ log π/β).
- Implicit policy constraint: AWR (weighting BC using exp(A/α)).
- Support constraints: BEAR-style MMD.
- TD-target modification: IQL with expectile regression to compute in-support V(s).

---

Below we add small reproducible demos illustrating key failure modes and how IQL / MMD constraints mitigate them.

In [ ]:
class OfflineDataset(Dataset):
    def __init__(self, states, actions, rewards, next_states, dones):
        self.states = torch.FloatTensor(states)
        self.actions = torch.FloatTensor(actions)
        self.rewards = torch.FloatTensor(rewards).unsqueeze(1)
        self.next_states = torch.FloatTensor(next_states)
        self.dones = torch.FloatTensor(dones).unsqueeze(1)

    def __len__(self):
        return len(self.states)

    def __getitem__(self, idx):
        return (self.states[idx], self.actions[idx], self.rewards[idx], 
                self.next_states[idx], self.dones[idx])

# Quick synthetic dataset for demos
STATE_DIM = 5
ACTION_DIM = 2  # continuous action
N_SAMPLES = 5000

# Synthetic data generating process (toy): linear policy + noise, reward is negative distance to goal
true_W = np.random.randn(STATE_DIM, ACTION_DIM)
states = np.random.randn(N_SAMPLES, STATE_DIM)
actions = states @ true_W + 0.1 * np.random.randn(N_SAMPLES, ACTION_DIM)
next_states = states + 0.1 * actions + 0.1 * np.random.randn(N_SAMPLES, STATE_DIM)
dones = np.zeros(N_SAMPLES)
# reward: higher when action aligns with a target linear function
target_action = (states @ true_W)
rewards = -np.linalg.norm(actions - target_action, axis=1)

dataset = OfflineDataset(states, actions, rewards, next_states, dones)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True)
print('Dataset prepared:', len(dataset))

## 3) Network templates: actor (Gaussian), Q-network, V-network, behavior model
These are small MLPs used by the agents below. Actor outputs Gaussian policy via mean and log std.

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, output_dim)
        )
    def forward(self, x):
        return self.net(x)

class GaussianActor(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=256, min_logstd=-10, max_logstd=2):
        super().__init__()
        self.mu_net = MLP(state_dim, action_dim, hidden)
        self.logstd = nn.Parameter(torch.zeros(action_dim))
        self.min_logstd = min_logstd
        self.max_logstd = max_logstd
    def get_distribution(self, states: torch.Tensor):
        mu = self.mu_net(states)
        logstd = torch.clamp(self.logstd, self.min_logstd, self.max_logstd)
        std = torch.exp(logstd)
        return Normal(mu, std)
    def forward(self, states: torch.Tensor):
        return self.get_distribution(states)

class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=256):
        super().__init__()
        self.net = MLP(state_dim + action_dim, 1, hidden)
    def forward(self, s, a):
        x = torch.cat([s, a], dim=-1)
        return self.net(x).squeeze(-1)

class VNetwork(nn.Module):
    def __init__(self, state_dim, hidden=256):
        super().__init__()
        self.net = MLP(state_dim, 1, hidden)
    def forward(self, s):
        return self.net(s).squeeze(-1)

# Behavior model (simple Gaussian conditional using MLP for mean and fixed std)
class BehaviorModel(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=256):
        super().__init__()
        self.mu_net = MLP(state_dim, action_dim, hidden)
        self.logstd = nn.Parameter(torch.zeros(action_dim))
    def get_distribution(self, s: torch.Tensor):
        mu = self.mu_net(s)
        std = torch.exp(self.logstd)
        return Normal(mu, std)

## 4) Advantage-Weighted Regression (AWR)
Weighted behavioral cloning where weights are exp(A / beta). Stable clipping on weights is recommended.

In [ ]:
class AWRAgent(nn.Module):
    def __init__(self, actor: GaussianActor, q_net: QNetwork, v_net: VNetwork, beta: float = 1.0):
        super().__init__()
        self.actor = actor
        self.q_net = q_net
        self.v_net = v_net
        self.beta = beta

    def compute_actor_loss(self, states, actions):
        # assume q_net and v_net are already trained or used as targets
        with torch.no_grad():
            q_vals = self.q_net(states, actions)  # (B,)
            v_vals = self.v_net(states)
            adv = (q_vals - v_vals).unsqueeze(-1)  # (B,1)
            weights = torch.exp(adv / self.beta)
            weights = torch.clamp(weights, max=20.0)
        dist = self.actor.get_distribution(states)
        logp = dist.log_prob(actions).sum(-1, keepdim=True)
        loss = -(weights * logp).mean()
        return loss

# Demo: initialize networks and do one AWR actor step
actor = GaussianActor(STATE_DIM, ACTION_DIM).to(device)
q_net = QNetwork(STATE_DIM, ACTION_DIM).to(device)
v_net = VNetwork(STATE_DIM).to(device)
awr = AWRAgent(actor, q_net, v_net, beta=1.0).to(device)
opt_actor = torch.optim.Adam(actor.parameters(), lr=3e-4)

# single batch step demo
batch = next(iter(dataloader))
states_b, actions_b = batch[0].to(device), batch[1].to(device)
loss_awr = awr.compute_actor_loss(states_b, actions_b)
opt_actor.zero_grad(); loss_awr.backward(); opt_actor.step()
print('AWR actor step completed, loss:', loss_awr.item())

In [ ]:
# Demo 1: Bootstrapping Error Amplification (toy discrete environment)
# We simulate a single-state MDP with two actions where only action 0 is present in dataset.
# Compare standard TD update (uses max_a) vs. IQL-style expectile target.

import copy

class ToyTabularEnv:
    def __init__(self):
        self.n_states = 1
        self.n_actions = 2
        # deterministic transitions back to same state
    def step(self, s, a):
        # reward: action 0 has reward 0, action 1 has reward 1 but is OOD (never observed)
        r = 0.0 if a == 0 else 1.0
        return 0, r, False, {}

# build dataset only with action 0 samples
toy = ToyTabularEnv()
N = 200
states_tb = np.zeros((N, 1), dtype=float)
actions_tb = np.zeros((N, 1), dtype=float)
rewards_tb = np.zeros((N,), dtype=float)
next_states_tb = np.zeros((N, 1), dtype=float)

# Q network: simple MLP over one-hot action input (state unused)
class TinyQ(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.lin = nn.Linear(n_actions, 1, bias=False)
    def forward(self, s, a_onehot):
        return self.lin(a_onehot).squeeze(-1)

# one-hot actions
def onehot(a, n):
    oh = np.zeros((len(a), n), dtype=float)
    for i, ai in enumerate(a): oh[i, int(ai)] = 1.0
    return oh

# prepare tensors
s_t = torch.FloatTensor(states_tb)
a_t = torch.LongTensor(actions_tb.flatten())
act_oh = torch.FloatTensor(onehot(actions_tb.flatten(), toy.n_actions))
r_t = torch.FloatTensor(rewards_tb)

# initialize two Q nets: TD (max backup) and IQL style (expectile-driven target)
q_td = TinyQ(toy.n_actions)
q_iql = TinyQ(toy.n_actions)
opt_td = torch.optim.Adam(q_td.parameters(), lr=1e-2)
opt_iql = torch.optim.Adam(q_iql.parameters(), lr=1e-2)

# training loop: TD uses max_a Q(next), IQL uses expectile value computed from observed actions (only action 0)
num_steps = 400
history = {'q_td_a0': [], 'q_td_a1': [], 'q_iql_a0': [], 'q_iql_a1': []}
for it in range(num_steps):
    # sample minibatch from dataset (all have action 0)
    idx = np.random.choice(N, size=64, replace=True)
    s_b = torch.FloatTensor(states_tb[idx])
    a_b = torch.LongTensor(actions_tb[idx].flatten())
    r_b = torch.FloatTensor(rewards_tb[idx])
    a_oh_b = torch.FloatTensor(onehot(a_b.numpy(), toy.n_actions))

    # TD update: target r + gamma * max_a' Q(next,a')
    with torch.no_grad():
        next_oh = torch.eye(toy.n_actions)
        q_next = q_td(s_b, next_oh)
        max_qnext = q_next.max().detach()
        target_td = r_b + 0.99 * max_qnext
    pred = q_td(s_b, a_oh_b)
    loss_td = F.mse_loss(pred, target_td)
    opt_td.zero_grad(); loss_td.backward(); opt_td.step()

    # IQL-style: compute expectile V (since dataset only has action 0, V is driven by Q(s,a=0))
    # simple empirical expectile: for scalar Q values of observed actions use gradient step
    with torch.no_grad():
        q_obs = q_iql(s_b, a_oh_b)
        # expectile tau>0.5 yields optimistic V in support
        tau = 0.7
        v = q_obs.mean()  # initial guess
        # one-step Newton-like update for expectile (small dataset) - simple: iterate few times
        for _ in range(5):
            diff = q_obs - v
            w = torch.where(diff > 0, tau, 1 - tau)
            v = (w * q_obs).sum() / w.sum()
    # TD-like Q update but target uses V(next)
    target_iql = r_b + 0.99 * v
    pred_iql = q_iql(s_b, a_oh_b)
    loss_iql = F.mse_loss(pred_iql, target_iql)
    opt_iql.zero_grad(); loss_iql.backward(); opt_iql.step()

    # record current Q estimates for both actions (evaluate on both one-hot vectors)
    with torch.no_grad():
        oh_all = torch.FloatTensor(np.eye(toy.n_actions))
        q_td_vals = q_td(s_b[:1], oh_all)
        q_iql_vals = q_iql(s_b[:1], oh_all)
        history['q_td_a0'].append(float(q_td_vals[0].item()))
        history['q_td_a1'].append(float(q_td_vals[1].item()))
        history['q_iql_a0'].append(float(q_iql_vals[0].item()))
        history['q_iql_a1'].append(float(q_iql_vals[1].item()))

# Plot results
plt.figure(figsize=(8,4))
plt.plot(history['q_td_a0'], label='TD Q(a=0)')
plt.plot(history['q_td_a1'], label='TD Q(a=1)')
plt.plot(history['q_iql_a0'], '--', label='IQL Q(a=0)')
plt.plot(history['q_iql_a1'], '--', label='IQL Q(a=1)')
plt.legend(); plt.title('Bootstrapping Error Demo: TD vs IQL (toy)'); plt.xlabel('Training step'); plt.ylabel('Q-value'); plt.show()

print('Final TD Q(a=1) vs IQL Q(a=1):', history['q_td_a1'][-1], history['q_iql_a1'][-1])

# Interpretation:
# In this toy example, standard TD backup can push up Q of OOD action (a=1) even though it never appears in dataset
# IQL approach that uses observed-action expectiles keeps the target within support and avoids large overestimation.

## 5) Implicit Q-Learning (IQL)
IQL uses expectile regression to fit V(s) from Q(s,a), then the Q targets use V(s') (in-support). Finally actor is learned by advantage-weighted regression.

In [ ]:
class IQLAgent(nn.Module):
    def __init__(self, actor: GaussianActor, q_net: QNetwork, v_net: VNetwork, expectile: float = 0.7, gamma: float = 0.99, temperature: float = 3.0):
        super().__init__()
        self.actor = actor
        self.q_net = q_net
        self.q_target = copy.deepcopy(q_net)
        self.v_net = v_net
        self.expectile = expectile
        self.gamma = gamma
        self.temperature = temperature

    def expectile_loss(self, diff):
        # diff = q - v
        weight = torch.where(diff > 0, self.expectile, 1.0 - self.expectile)
        return (weight * (diff ** 2)).mean()

    def train_step(self, batch, opt_v, opt_q, opt_actor, tau=0.005):
        states, actions, rewards, next_states, dones = [b.to(device) for b in batch]
        # 1) V update (expectile regression)
        with torch.no_grad():
            q_pred = self.q_target(states, actions)
        v_pred = self.v_net(states)
        v_loss = self.expectile_loss(q_pred - v_pred)
        opt_v.zero_grad(); v_loss.backward(); opt_v.step()

        # 2) Q update (TD with V target)
        with torch.no_grad():
            next_v = self.v_net(next_states)
            target_q = rewards.squeeze(-1) + self.gamma * (1.0 - dones.squeeze(-1)) * next_v
        q_pred_now = self.q_net(states, actions)
        q_loss = F.mse_loss(q_pred_now, target_q)
        opt_q.zero_grad(); q_loss.backward(); opt_q.step()

        # soft-update q_target
        for p, pt in zip(self.q_net.parameters(), self.q_target.parameters()):
            pt.data.copy_(tau * p.data + (1 - tau) * pt.data)

        # 3) Actor update (AWR extraction)
        with torch.no_grad():
            q_val = self.q_target(states, actions)
            v_val = self.v_net(states)
            adv = (q_val - v_val).unsqueeze(-1)
            weights = torch.exp(adv * self.temperature)
            weights = torch.clamp(weights, max=100.0)
        dist = self.actor.get_distribution(states)
        logp = dist.log_prob(actions).sum(-1, keepdim=True)
        actor_loss = -(weights * logp).mean()
        opt_actor.zero_grad(); actor_loss.backward(); opt_actor.step()

        return v_loss.item(), q_loss.item(), actor_loss.item()

# Demo: initialize IQL agent and optimizers
actor_iql = GaussianActor(STATE_DIM, ACTION_DIM).to(device)
q_net_iql = QNetwork(STATE_DIM, ACTION_DIM).to(device)
v_net_iql = VNetwork(STATE_DIM).to(device)
iql = IQLAgent(actor_iql, q_net_iql, v_net_iql)
opt_v = torch.optim.Adam(v_net_iql.parameters(), lr=3e-4)
opt_q = torch.optim.Adam(q_net_iql.parameters(), lr=3e-4)
opt_actor = torch.optim.Adam(actor_iql.parameters(), lr=3e-4)

# single training step demo
batch = next(iter(dataloader))
v_l, q_l, a_l = iql.train_step(batch, opt_v, opt_q, opt_actor)
print('IQL step: v_loss=%.4f q_loss=%.4f actor_loss=%.4f' % (v_l, q_l, a_l))

## 6) Explicit Policy Constraint: KL penalty and MMD (BEAR-style)
- KL constraint: penalize KL(pi || beta) where beta is a behavior model learned from data.
- MMD: ensure samples from pi are close to behavior samples in RKHS (support constraint).

In [ ]:
class ExplicitConstraintAgent(nn.Module):
    def __init__(self, actor: GaussianActor, critic: QNetwork, behavior_model: BehaviorModel, lambda_k: float = 0.5):
        super().__init__()
        self.actor = actor
        self.critic = critic
        self.behavior = behavior_model
        self.lambda_k = lambda_k

    def train_actor_kl(self, states, opt_actor):
        pi = self.actor.get_distribution(states)
        actions_pi = pi.rsample()
        q_values = self.critic(states, actions_pi)
        beta = self.behavior.get_distribution(states)
        # compute element-wise KL
        kl = torch.distributions.kl.kl_divergence(pi, beta).sum(-1, keepdim=True)
        loss = - (q_values.unsqueeze(-1) - self.lambda_k * kl).mean()
        opt_actor.zero_grad(); loss.backward(); opt_actor.step()
        return loss.item()

# MMD loss
def mmd_loss(samples_pi: torch.Tensor, samples_beta: torch.Tensor, sigma: float = 20.0) -> torch.Tensor:
    # samples: (N, action_dim)
    diff_pi_pi = samples_pi.unsqueeze(1) - samples_pi.unsqueeze(0)
    k_pi_pi = torch.exp(-torch.sum(diff_pi_pi**2, dim=-1) / (2.0 * sigma))

    diff_beta_beta = samples_beta.unsqueeze(1) - samples_beta.unsqueeze(0)
    k_beta_beta = torch.exp(-torch.sum(diff_beta_beta**2, dim=-1) / (2.0 * sigma))

    diff_pi_beta = samples_pi.unsqueeze(1) - samples_beta.unsqueeze(0)
    k_pi_beta = torch.exp(-torch.sum(diff_pi_beta**2, dim=-1) / (2.0 * sigma))

    mmd = k_pi_pi.mean() + k_beta_beta.mean() - 2.0 * k_pi_beta.mean()
    return mmd

# Demo: train explicit constraint agent for one step
behavior = BehaviorModel(STATE_DIM, ACTION_DIM).to(device)
# fit behavior model to dataset quickly (MLE)
opt_beh = torch.optim.Adam(behavior.parameters(), lr=1e-3)
for epoch in range(5):
    for batch in dataloader:
        s_b, a_b = batch[0].to(device), batch[1].to(device)
        dist = behavior.get_distribution(s_b)
        loss = -dist.log_prob(a_b).sum(-1).mean()
        opt_beh.zero_grad(); loss.backward(); opt_beh.step()

critic = QNetwork(STATE_DIM, ACTION_DIM).to(device)
actor_ec = GaussianActor(STATE_DIM, ACTION_DIM).to(device)
exp_agent = ExplicitConstraintAgent(actor_ec, critic, behavior, lambda_k=0.5)
opt_actor_ec = torch.optim.Adam(actor_ec.parameters(), lr=3e-4)

# One training step: sample states, compute KL or MMD regularized loss
s_b, a_b, r_b, ns_b, d_b = next(iter(dataloader))
s_b = s_b.to(device)
kl_loss = exp_agent.train_actor_kl(s_b, opt_actor_ec)

# MMD demonstration: sample actions from actor and from data
with torch.no_grad():
    dist_pi = actor_ec.get_distribution(s_b)
    samples_pi = dist_pi.rsample()
    samples_beta = a_b.to(device)
mmd = mmd_loss(samples_pi, samples_beta, sigma=10.0)
print('Explicit KL step loss:', kl_loss, 'MMD:', mmd.item())

In [ ]:
# Demo 2: BEAR-style MMD constraint effect on actor
# We train an unconstrained actor to maximize a fixed critic's Q and compare with an MMD-penalized actor.

# Reuse synthetic dataset and critic initialized earlier
critic = QNetwork(STATE_DIM, ACTION_DIM).to(device)
# random critic for demonstration (we won't train it for clarity)
for p in critic.parameters():
    p.requires_grad = False

# actor unconstrained and actor with MMD penalty
actor_uc = GaussianActor(STATE_DIM, ACTION_DIM).to(device)
actor_mmd = GaussianActor(STATE_DIM, ACTION_DIM).to(device)
opt_uc = torch.optim.Adam(actor_uc.parameters(), lr=1e-3)
opt_mmd = torch.optim.Adam(actor_mmd.parameters(), lr=1e-3)

# sample a reference batch from dataset as behavior samples
s_ref, a_ref, _, _, _ = next(iter(dataloader))
s_ref = s_ref.to(device); a_ref = a_ref.to(device)

mmd_history = {'unconstrained': [], 'mmd_actor': []}
q_history = {'unconstrained': [], 'mmd_actor': []}
for it in range(200):
    # unconstrained actor: update to maximize critic(s, a) expectation
    dist_uc = actor_uc.get_distribution(s_ref)
    samples_uc = dist_uc.rsample()
    q_uc = critic(s_ref, samples_uc).mean()
    loss_uc = -q_uc
    opt_uc.zero_grad(); loss_uc.backward(); opt_uc.step()

    # mmd actor: maximize Q but penalize MMD to behavior actions
    dist_m = actor_mmd.get_distribution(s_ref)
    samples_m = dist_m.rsample()
    q_m = critic(s_ref, samples_m).mean()
    mmd_val = mmd_loss(samples_m, a_ref, sigma=20.0)
    loss_mmd = -q_m + 10.0 * mmd_val
    opt_mmd.zero_grad(); loss_mmd.backward(); opt_mmd.step()

    if it % 10 == 0:
        mmd_history['unconstrained'].append(mmd_loss(samples_uc, a_ref, sigma=20.0).item())
        mmd_history['mmd_actor'].append(mmd_val.item())
        q_history['unconstrained'].append(q_uc.item())
        q_history['mmd_actor'].append(q_m.item())

# Plot MMD and Q values
plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.plot(mmd_history['unconstrained'], label='unconstrained MMD'); plt.plot(mmd_history['mmd_actor'], label='mmd_actor'); plt.title('MMD to behavior'); plt.legend()
plt.subplot(1,2,2); plt.plot(q_history['unconstrained'], label='unconstrained Q'); plt.plot(q_history['mmd_actor'], label='mmd_actor'); plt.title('Mean critic Q'); plt.legend()
plt.tight_layout(); plt.show()

print('Final MMD (unconstrained vs mmd_actor):', mmd_history['unconstrained'][-1], mmd_history['mmd_actor'][-1])

# Interpretation: unconstrained actor can seek high Q but drift far from dataset (high MMD), while MMD penalty keeps policy in-support at cost of some Q.

## 7) Practical tips & diagnostics
- Clip AWR/IQL weights to avoid exploding gradients.
- Monitor ESS when using importance sampling (sample-based IRL methods).
- For GCL-style adversarial learning, prefer PPO/TRPO for stable policy updates; REINFORCE is simple but high-variance.
- When using MMD, choose kernel bandwidth carefully (median heuristic) and normalize actions.

---

If you'd like, I can also:
- Add unit tests for each agent (train a few steps and assert loss decreases), or
- Add a small script to run hyperparameter sweeps and plot results.

Tell me which next step you prefer. ✅

## 8) Theoretical notes and practical heuristics

- **Bootstrapping error amplification:** small extrapolation errors on OOD actions can be amplified by repeated Bellman backups; prefer algorithms that keep targets inside the data support (IQL) or penalize policy divergence (AWR/KL/MMD).

- **Expectile intuition:** the expectile with τ>0.5 is like a tilted mean that emphasizes higher Q-values observed in data, yielding an optimistic-but-in-support V(s) used to form stable TD targets.

- **MMD & kernel choice:** kernel bandwidth (σ) matters; use median heuristic on pairwise distances or tune.

- **Practical tips:**
  - Clip exponentiated weights (AWR / IQL actor extraction) to avoid exploding gradients.
  - Monitor ESS when using importance-sampling estimators.
  - Prefer trust-region style policy updates (PPO/TRPO) when doing adversarial or policy-improvement steps.

---

If you want, I can: add LaTeX equations inline, include a small unit test suite that runs one training step for each agent and asserts loss decreases, or add a short README and references. Which would you like next?